# Story 2: polarity-dependent topographic response

Select strongly topographic eddies that illustrate the asymmetric response found in the population analysis. CEs are ranked for coherent opposition to the signed total PV gradient; AEs are ranked for loss of a single coherent PV-relative direction. The AE category is therefore a disrupted-response example, not a claim that every topographic AE is random.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ROOT is None: raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or a subfolder.')
for path in (ROOT, ROOT / 'case_studies'):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
from paper_case_study_tools import PaperCaseConfig, plot_paper_case, rank_topographic_response_cases, select_ranked_cases
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 40)

## Load and classify

The primary topographic regime requires the smoothed topographic PV-gradient contribution to exceed the planetary contribution by at least 2:1 and core-mean depth to be no greater than 2,000 m.

In [ ]:
config = PaperCaseConfig()
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, _ = tilt.load_tilt_tables(paths)
df_eddies = tilt.add_region_labels(df_eddies, grid)
df_eddies = tilt.add_pv_gradient_terms(df_eddies, grid, core_mean=True)
df_case, ranking = rank_topographic_response_cases(df_eddies, config)
display(ranking.groupby('response_type')['eligible'].agg(candidates='size', eligible='sum'))

## Population context

Plot the raw signed-PV angular separation as well as the corrected polarity-aware preference error. This makes the convention explicit and shows how much broader the AE response is.

In [ ]:
use = df_case[df_case.topographic_regime & df_case.direction_valid]
bins = np.arange(0, 181, 10)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for cyc, color in [('AE','firebrick'),('CE','royalblue')]:
    q = use[use.Cyc.eq(cyc)]
    axes[0].hist(q.dtheta_PV_grad, bins=bins, density=True, histtype='step', lw=2, color=color, label=f'{cyc} (n={len(q):,})')
    axes[1].hist(q.preference_error_deg, bins=bins, density=True, histtype='step', lw=2, color=color, label=cyc)
axes[0].set(title='Raw |tilt - signed PV gradient|', xlabel='Angular separation (deg)', ylabel='Density')
axes[1].set(title='Corrected polarity-aware response', xlabel='Preference error (deg)')
for ax in axes: ax.legend(frameon=False)
fig.suptitle('Strongly topographic observations');

## Ranked candidates

CE scores reward sustained low preference error. AE scores reward low circular resultant length of the PV-relative angle, meaning the eddy samples multiple relative directions rather than remaining consistently aligned or opposed.

In [ ]:
columns = ['Eddy','Cyc','Region','response_type','case_score','lifetime_days','regime_observations','regime_fraction','longest_regime_run','matching_fraction','longest_matching_run','median_preference_error_deg','relative_direction_resultant','response_quality','directional_coverage','median_tilt_km','median_depth_m']
for response in ranking.response_type.unique():
    print(f'\n{response}')
    display(ranking.loc[(ranking.response_type.eq(response)) & ranking.eligible, columns].head(20).round(3))

In [ ]:
selected = select_ranked_cases(ranking, 'response_type', n_per_group=3)
selected

## Candidate figures

Orange background shading marks qualifying topographic days. Prefer coherent shelf/slope encounters over tracks dominated by coastline clipping or repeated detection jumps.

In [ ]:
for response, eddies in selected.items():
    for eddy_id in eddies:
        track = df_case[df_case.Eddy.eq(eddy_id)]
        plot_paper_case(track, grid, config=config, title=f'{response} - eddy {eddy_id}')
        plt.show()

## Rossby-number context

Rossby number is shown as a modifier, not an independent topographic mechanism: it enters the topographic PV term through relative vorticity. Use this table to identify high-rotation candidates for discussion, but do not interpret a raw Ro association as independent evidence.

In [ ]:
selected_ids = [eddy for values in selected.values() for eddy in values]
audit = df_case[df_case.Eddy.isin(selected_ids) & df_case.topographic_regime].groupby(['Cyc','Eddy']).agg(observations=('Day','size'), median_Ro=('Ro','median'), max_Ro=('Ro','max'), median_tilt_km=('TiltDis','median'), median_ratio=('topo_plan_ratio_smooth','median'))
display(audit.round(3))

## Interpretation guardrails

The CE case should show a coherent response as the signed PV gradient follows topography. The AE case should show genuine directional disruption across a sustained topographic episode, not simply noisy bearings produced by very small tilt. Ro is a mathematical amplifier of the topographic term and should be described accordingly.